# Day 021 | Pandas — GroupBy & Aggregation

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `.groupby()` | Split data into groups | Group layers by soil type or borehole |
| `.agg()` | Apply multiple functions at once | Mean N-value AND max depth per soil type |
| `.transform()` | Group calculation, keep original shape | Normalize N-value within each borehole |
| `.apply()` | Custom group function | Classify each borehole's bearing capacity |
| `pivot_table()` | Summarize data in matrix form | Soil type vs borehole summary table |
| `pd.crosstab()` | Count combinations | How many sand layers per borehole? |

---

## Our Geotechnical Problem

A site investigation for a highway bridge has **4 boreholes** drilled to 15 m depth. Each borehole logged 10 layers with SPT N-values, soil types, and lab results.

The geotechnical engineer needs to answer:

- What is the **average N-value** for each soil type across the site?
- Which borehole has the **weakest** layers?
- How does **water content** vary between clay and sand?
- What is the **site-wide summary** comparing all 4 boreholes?

All of these require **GroupBy** — splitting data into groups, computing statistics per group, and combining the results.

---

## Part 1: Loading Site Investigation Data

We have a CSV with 40 rows of borehole data from 4 boreholes (BH-01 through BH-04), each with 10 layers at 1.5 m intervals down to 15 m.

In [ ]:
# ─────────────────────────────────────────────────────
# LOADING SITE INVESTIGATION DATA
# ─────────────────────────────────────────────────────

import pandas as pd
import numpy as np

df = pd.read_csv('site_investigation.csv')

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Boreholes: {df['borehole_id'].unique().tolist()}")
print(f"Soil types: {df['soil_type'].unique().tolist()}")
print()
df.head(12)

---

## Part 2: Basic GroupBy — Split-Apply-Combine

The **GroupBy** pattern has three steps:

1. **Split** — Divide the DataFrame into groups (by soil type, by borehole, etc.)
2. **Apply** — Compute something for each group (mean, max, count, custom function)
3. **Combine** — Merge results back into a single output

```python
df.groupby('soil_type')['N_value'].mean()
#   ↑ split by soil_type  ↑ select column  ↑ apply mean
```

In [ ]:
# ─────────────────────────────────────────────────────
# BASIC GROUPBY — Mean N-value per soil type
# ─────────────────────────────────────────────────────

# Average N-value by soil type
n_by_soil = df.groupby('soil_type')['N_value'].mean()
print("Mean N-value by soil type:")
print(n_by_soil.round(1).sort_values(ascending=False))
print()

# Average N-value by borehole
n_by_bh = df.groupby('borehole_id')['N_value'].mean()
print("Mean N-value by borehole:")
print(n_by_bh.round(1))
print()

# Count of layers per soil type
count_by_soil = df.groupby('soil_type')['N_value'].count()
print("Layer count by soil type:")
print(count_by_soil.sort_values(ascending=False))

In [ ]:
# ─────────────────────────────────────────────────────
# MULTIPLE AGGREGATIONS ON ONE COLUMN
# ─────────────────────────────────────────────────────

# Multiple statistics for N-value by soil type
n_stats = df.groupby('soil_type')['N_value'].agg(['mean', 'median', 'min', 'max', 'std', 'count'])
n_stats = n_stats.round(1)

print("N-value statistics by soil type:")
print("=" * 65)
print(n_stats.to_string())
print()
print("Observation: Bedrock has highest N (>54), Fill has lowest (<5).")
print("This matches engineering expectations — N increases with stiffness.")

---

## Part 3: Multi-Column Aggregation — `.agg()`

`.agg()` lets you apply **different functions to different columns** in a single call:

```python
df.groupby('soil_type').agg({
    'N_value': ['mean', 'std'],     # mean and std of N
    'water_content': 'mean',        # just mean of water content
    'depth_m': 'max'                # max depth where this soil appears
})
```

In [ ]:
# ─────────────────────────────────────────────────────
# MULTI-COLUMN AGGREGATION
# ─────────────────────────────────────────────────────

# Different aggregations for different columns
soil_summary = df.groupby('soil_type').agg({
    'N_value': ['mean', 'std', 'count'],
    'water_content': ['mean', 'min', 'max'],
    'unit_weight': 'mean',
    'fines_content': 'mean',
    'depth_m': ['min', 'max']
}).round(1)

print("Comprehensive soil type summary:")
print("=" * 80)
print(soil_summary.to_string())
print()
print("Key insight: Clay has highest water content (avg ~31%) and fines (~91%).")
print("Sand has the widest depth range, appearing throughout the profile.")

In [ ]:
# ─────────────────────────────────────────────────────
# NAMED AGGREGATION — Cleaner column names
# ─────────────────────────────────────────────────────

# Named aggregation gives you control over output column names
bh_summary = df.groupby('borehole_id').agg(
    avg_N=('N_value', 'mean'),
    min_N=('N_value', 'min'),
    max_N=('N_value', 'max'),
    max_depth=('depth_m', 'max'),
    avg_water=('water_content', 'mean'),
    avg_unit_wt=('unit_weight', 'mean'),
    n_layers=('N_value', 'count')
).round(1)

print("Borehole summary (named aggregation):")
print("=" * 70)
print(bh_summary.to_string())
print()

# Which borehole is weakest?
weakest = bh_summary['avg_N'].idxmin()
strongest = bh_summary['avg_N'].idxmax()
print(f"Weakest borehole:   {weakest} (avg N = {bh_summary.loc[weakest, 'avg_N']})")
print(f"Strongest borehole: {strongest} (avg N = {bh_summary.loc[strongest, 'avg_N']})")

---

## Part 4: GroupBy with Multiple Keys

You can group by **multiple columns** to get finer-grained analysis — for example, the average N-value for each soil type **within each borehole**.

In [ ]:
# ─────────────────────────────────────────────────────
# GROUPBY WITH MULTIPLE KEYS
# ─────────────────────────────────────────────────────

# Average N-value per soil type PER borehole
multi_group = df.groupby(['borehole_id', 'soil_type']).agg(
    avg_N=('N_value', 'mean'),
    layers=('N_value', 'count'),
    avg_wc=('water_content', 'mean')
).round(1)

print("N-value by borehole AND soil type:")
print("=" * 55)
print(multi_group.to_string())
print()

# Access a specific group
print("Sand layers in BH-03:")
bh03_sand = df[(df['borehole_id'] == 'BH-03') & (df['soil_type'] == 'Sand')]
print(bh03_sand[['depth_m', 'N_value', 'water_content']].to_string(index=False))

---

## Part 5: Transform — Group Calculation, Original Shape

`.transform()` computes a value per group but returns a Series with the **same length** as the original DataFrame. This is perfect for:

- Adding a "group mean" column
- Normalizing values within each group
- Flagging rows that deviate from their group

```python
# Each row gets the mean N of its borehole:
df['bh_mean_N'] = df.groupby('borehole_id')['N_value'].transform('mean')
```

In [ ]:
# ─────────────────────────────────────────────────────
# TRANSFORM — Add group statistics to each row
# ─────────────────────────────────────────────────────

# Add the borehole-average N to each row
df['bh_mean_N'] = df.groupby('borehole_id')['N_value'].transform('mean').round(1)

# Calculate how each layer compares to its borehole average
df['N_vs_avg'] = (df['N_value'] - df['bh_mean_N']).round(1)

print("Each layer vs its borehole average:")
print(df[['borehole_id', 'depth_m', 'N_value', 'bh_mean_N', 'N_vs_avg']].head(15).to_string())
print()
print("Negative N_vs_avg = weaker than borehole average")
print("Positive N_vs_avg = stronger than borehole average")

In [ ]:
# ─────────────────────────────────────────────────────
# TRANSFORM — Normalize N-values within each borehole
# ─────────────────────────────────────────────────────

# Z-score normalization within each borehole
# (how many standard deviations from the borehole mean?)
def zscore(x):
    return (x - x.mean()) / x.std()

df['N_zscore'] = df.groupby('borehole_id')['N_value'].transform(zscore).round(2)

# Flag anomalous layers (|z| > 1.5)
df['anomalous'] = df['N_zscore'].abs() > 1.5

print("Z-score normalized N-values (within each borehole):")
print(df[['borehole_id', 'depth_m', 'N_value', 'N_zscore', 'anomalous']].head(15).to_string())
print()

anomalous = df[df['anomalous']]
print(f"Anomalous layers (|z| > 1.5): {len(anomalous)}")
if len(anomalous) > 0:
    print(anomalous[['borehole_id', 'depth_m', 'N_value', 'soil_type', 'N_zscore']].to_string())

---

## Part 6: Apply — Custom Group Functions

`.apply()` lets you write **any custom function** that receives a group DataFrame and returns whatever you want. This is the most flexible GroupBy operation.

```python
def classify_borehole(group):
    avg_n = group['N_value'].mean()
    if avg_n < 15: return 'Weak'
    elif avg_n < 30: return 'Medium'
    else: return 'Strong'

df.groupby('borehole_id').apply(classify_borehole)
```

In [ ]:
# ─────────────────────────────────────────────────────
# APPLY — Custom borehole classification
# ─────────────────────────────────────────────────────

def borehole_report(group):
    """Generate a summary for one borehole."""
    avg_n = group['N_value'].mean()
    weak_layers = (group['N_value'] < 10).sum()
    dominant_soil = group['soil_type'].value_counts().index[0]
    
    if avg_n < 20:
        rating = 'Weak'
    elif avg_n < 35:
        rating = 'Medium'
    else:
        rating = 'Strong'
    
    return pd.Series({
        'avg_N': round(avg_n, 1),
        'weak_layers': weak_layers,
        'dominant_soil': dominant_soil,
        'rating': rating,
        'max_depth': group['depth_m'].max(),
        'n_soil_types': group['soil_type'].nunique()
    })

bh_report = df.groupby('borehole_id').apply(borehole_report)
print("Custom borehole classification:")
print("=" * 65)
print(bh_report.to_string())
print()
print("Engineer's note: All boreholes reach 15m. BH-04 has the most")
print("weak layers (3) and lowest avg N — may need deeper foundation.")

---

## Part 7: Pivot Tables — Matrix Summaries

A **pivot table** reshapes data into a matrix: rows = one grouping, columns = another, values = aggregation.

```python
pd.pivot_table(df,
    values='N_value',          # what to summarize
    index='soil_type',         # row labels
    columns='borehole_id',     # column labels
    aggfunc='mean'             # how to summarize
)
```

This creates a soil-type-by-borehole matrix — exactly what an engineer would put in a report.

In [ ]:
# ─────────────────────────────────────────────────────
# PIVOT TABLE — Soil type vs Borehole
# ─────────────────────────────────────────────────────

# Mean N-value: soil type (rows) x borehole (columns)
pivot_n = pd.pivot_table(
    df,
    values='N_value',
    index='soil_type',
    columns='borehole_id',
    aggfunc='mean'
).round(1)

print("Mean N-value: Soil Type vs Borehole")
print("=" * 55)
print(pivot_n.to_string())
print()

# Mean water content: soil type (rows) x borehole (columns)
pivot_wc = pd.pivot_table(
    df,
    values='water_content',
    index='soil_type',
    columns='borehole_id',
    aggfunc='mean'
).round(1)

print("\nMean Water Content (%): Soil Type vs Borehole")
print("=" * 55)
print(pivot_wc.to_string())
print()
print("NaN means that soil type was not encountered in that borehole.")

In [ ]:
# ─────────────────────────────────────────────────────
# PIVOT TABLE WITH MARGINS — Row/column totals
# ─────────────────────────────────────────────────────

# Pivot with margins (All row/column = overall averages)
pivot_full = pd.pivot_table(
    df,
    values='N_value',
    index='soil_type',
    columns='borehole_id',
    aggfunc='mean',
    margins=True,
    margins_name='Site Avg'
).round(1)

print("Mean N-value with Site Averages:")
print("=" * 65)
print(pivot_full.to_string())
print()
print("The 'Site Avg' row shows each borehole's overall strength.")
print("The 'Site Avg' column shows each soil type's site-wide average.")

---

## Part 8: Cross-Tabulation — Counting Combinations

`pd.crosstab()` counts how many times each combination occurs. For example: how many layers of each soil type appear in each borehole?

In [ ]:
# ─────────────────────────────────────────────────────
# CROSS-TABULATION — Count soil layers per borehole
# ─────────────────────────────────────────────────────

# Count: soil type x borehole
xtab = pd.crosstab(df['soil_type'], df['borehole_id'], margins=True)

print("Layer count: Soil Type vs Borehole")
print("=" * 50)
print(xtab.to_string())
print()

# Add a density classification column, then cross-tab
def classify_density(n):
    if n < 4: return '1-Very Loose'
    elif n < 10: return '2-Loose'
    elif n < 30: return '3-Medium Dense'
    elif n < 50: return '4-Dense'
    else: return '5-Very Dense'

df['density_class'] = df['N_value'].apply(classify_density)

xtab_density = pd.crosstab(df['density_class'], df['borehole_id'], margins=True)
print("\nDensity class distribution by borehole:")
print("=" * 50)
print(xtab_density.to_string())
print()
print("BH-03 has the most Very Dense layers (3).")
print("BH-04 has the most Loose layers (3) — weaker profile.")

---

## Part 9: Complete Application — Site-Wide Geotechnical Summary

Let us combine all GroupBy techniques to produce a comprehensive site investigation summary report, comparing soil conditions across all 4 boreholes.

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE: SITE INVESTIGATION SUMMARY REPORT
# ══════════════════════════════════════════════════════

print("╔" + "═" * 68 + "╗")
print("║  SITE INVESTIGATION SUMMARY — HIGHWAY BRIDGE PROJECT              ║")
print("╚" + "═" * 68 + "╝")

# ── Section 1: Borehole Overview ──
print("\n1. BOREHOLE OVERVIEW")
print("─" * 70)

overview = df.groupby('borehole_id').agg(
    depth_range=('depth_m', lambda x: f"{x.min():.1f}-{x.max():.1f}"),
    avg_N=('N_value', 'mean'),
    min_N=('N_value', 'min'),
    max_N=('N_value', 'max'),
    n_soil_types=('soil_type', 'nunique'),
    avg_wc=('water_content', 'mean')
).round(1)

print(overview.to_string())

# ── Section 2: Soil Type Statistics ──
print("\n\n2. SOIL TYPE STATISTICS (SITE-WIDE)")
print("─" * 70)

soil_stats = df.groupby('soil_type').agg(
    count=('N_value', 'count'),
    avg_N=('N_value', 'mean'),
    std_N=('N_value', 'std'),
    avg_wc=('water_content', 'mean'),
    avg_gamma=('unit_weight', 'mean'),
    avg_fines=('fines_content', 'mean'),
    depth_min=('depth_m', 'min'),
    depth_max=('depth_m', 'max')
).round(1)

print(soil_stats.to_string())

# ── Section 3: N-value Matrix ──
print("\n\n3. MEAN N-VALUE MATRIX (Soil Type x Borehole)")
print("─" * 70)

pivot = pd.pivot_table(
    df, values='N_value', index='soil_type',
    columns='borehole_id', aggfunc='mean', margins=True,
    margins_name='Average'
).round(1)

print(pivot.to_string())

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE (continued): Engineering Interpretation
# ══════════════════════════════════════════════════════

# ── Section 4: Weak Layer Identification ──
print("4. WEAK LAYER IDENTIFICATION (N < 10)")
print("─" * 70)

weak = df[df['N_value'] < 10].copy()
weak_summary = weak.groupby('borehole_id').agg(
    n_weak_layers=('N_value', 'count'),
    min_N=('N_value', 'min'),
    weak_soils=('soil_type', lambda x: ', '.join(x.unique())),
    max_weak_depth=('depth_m', 'max')
)

print(weak_summary.to_string())

# ── Section 5: Consistency Profile ──
print("\n\n5. DENSITY CLASSIFICATION DISTRIBUTION")
print("─" * 70)

density_dist = pd.crosstab(
    df['density_class'], df['borehole_id'],
    margins=True
)
print(density_dist.to_string())

# ── Section 6: Engineering Recommendations ──
print("\n\n6. ENGINEERING SUMMARY")
print("─" * 70)

site_avg_n = df['N_value'].mean()
weakest_bh = df.groupby('borehole_id')['N_value'].mean().idxmin()
strongest_bh = df.groupby('borehole_id')['N_value'].mean().idxmax()
dominant_soil = df['soil_type'].value_counts().index[0]
deepest_weak = df[df['N_value'] < 10]['depth_m'].max()

print(f"  Site average N-value:    {site_avg_n:.1f}")
print(f"  Weakest borehole:        {weakest_bh}")
print(f"  Strongest borehole:      {strongest_bh}")
print(f"  Dominant soil type:      {dominant_soil}")
print(f"  Deepest weak layer:      {deepest_weak:.1f} m")
print(f"  Total boreholes:         {df['borehole_id'].nunique()}")
print(f"  Total layers logged:     {len(df)}")
print("═" * 70)

# Clean up helper columns
df.drop(columns=['bh_mean_N', 'N_vs_avg', 'N_zscore', 'anomalous', 'density_class'],
        inplace=True, errors='ignore')

---

## Practice Exercises

### Exercise 1: GroupBy statistics
Group the data by `borehole_id` and compute the **coefficient of variation** (std/mean) for N-value. Which borehole has the most variable soil profile?

### Exercise 2: Pivot table
Create a pivot table showing the **mean water content** by soil type (rows) and borehole (columns). Which soil type has the most consistent water content across boreholes?

### Exercise 3: Custom apply
Write a function that takes a borehole group and returns the depth at which N first exceeds 30 (competent bearing stratum). Apply it to all boreholes.

In [ ]:
# EXERCISE 1
# cv = df.groupby('borehole_id')['N_value'].agg(lambda x: x.std() / x.mean())
# print(cv.round(3))
# print(f"Most variable: {cv.idxmax()} (CV = {cv.max():.3f})")

In [ ]:
# EXERCISE 2
# pivot_wc = pd.pivot_table(df, values='water_content',
#     index='soil_type', columns='borehole_id', aggfunc='mean')
# pivot_wc['range'] = pivot_wc.max(axis=1) - pivot_wc.min(axis=1)
# print(pivot_wc.round(1))

In [ ]:
# EXERCISE 3
# def depth_to_competent(group):
#     competent = group[group['N_value'] >= 30]
#     if len(competent) > 0:
#         return competent['depth_m'].min()
#     return None
#
# depths = df.groupby('borehole_id').apply(depth_to_competent)
# print(depths)

---

## Key Takeaways

1. **GroupBy** follows the split-apply-combine pattern: `df.groupby('column').agg(func)`. In geotechnics, you group by borehole ID, soil type, or depth interval constantly.

2. **`.agg()`** with named aggregation gives clean, multi-statistic summaries in one call — perfect for engineering reports.

3. **`.transform()`** returns data at the original row level — use it to normalize N-values within each borehole or flag anomalous layers relative to their group.

4. **Pivot tables** create matrix-format summaries (soil type vs borehole) that map directly to geotechnical report tables.

5. **Cross-tabulation** counts category combinations — how many sand layers per borehole, density class distributions, etc.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| `.groupby().mean()` | Feature engineering — group-level statistics as features |
| `.transform()` | `sklearn` group-wise normalization, target encoding |
| `.agg()` multi-stats | Creating multiple features from one column per group |
| `.apply()` custom func | Custom feature extraction per sample group |
| Pivot tables | Confusion matrices, feature correlation heatmaps |
| Cross-tabulation | `sklearn.metrics.confusion_matrix()` |
| Group-based anomaly detection | Isolation Forest per soil type |
| Coefficient of variation | Feature importance ranking |

In ML for geotechnics, GroupBy is your primary **feature engineering** tool. Creating group-level statistics (mean N per soil type, max depth per borehole) as input features dramatically improves model performance.

---

*Day 021 of 100 — GroupBy turns raw logs into engineering insights!*